In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql import functions as F
import country_converter as coco
from itertools import chain

CLEANED_DATA_DIR = Path("../data/cleaned")
PROCESSED_DATA_DIR = Path("../data")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet(f"{CLEANED_DATA_DIR}/4c_eea_co2_emissions_from_passenger_cars-001.parquet")

df.show(10)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl
import pyspark.sql.functions as F
from torch.utils.data import TensorDataset, DataLoader
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from VehicleAutoencoder import VehicleAutoencoder

def prepare_vehicle_data(
    spark_df, 
    train_val_split=0.8, 
    seed=42, 
    device="cuda"
):
    feature_cols = [
        "mass_in_running_order (kg)",
        "co2_emissions_WLTP (g/km)",
        "engine_capacity (cm3)",
        "engine_power (KW)",
        "electric_energy_consumption (Wh/km)"
    ]
    
    metadata_cols = [
        "geo", 
        "TIME_PERIOD", 
        "registrations", 
        "manufacturer_name_eu_standard_denomination", 
        "commercial_name", 
        "variant", 
        "Motor energy"
    ]

    df_clean = spark_df.dropna(subset=feature_cols)

    motor_col = F.lower(F.col("Motor energy"))
    is_electric = motor_col.isin(["electric", "bev", "pure electric"])
    is_ice = motor_col.isin(["petrol", "diesel", "cng", "lpg", "petrol/cng"])

    df_zeroed = df_clean.withColumn(
        "co2_emissions_WLTP (g/km)",
        F.when(is_electric, 0.0).otherwise(F.col("co2_emissions_WLTP (g/km)"))
    ).withColumn(
        "engine_capacity (cm3)",
        F.when(is_electric, 0.0).otherwise(F.col("engine_capacity (cm3)"))
    ).withColumn(
        "electric_energy_consumption (Wh/km)",
        F.when(is_ice, 0.0).otherwise(F.col("electric_energy_consumption (Wh/km)"))
    )

    pdf = df_zeroed.select(metadata_cols + feature_cols).toPandas()

    metadata_df = pdf[metadata_cols].copy()
    
    X_raw = pdf[feature_cols].values.astype(np.float32)
    
    mean = np.mean(X_raw, axis=0)
    std = np.std(X_raw, axis=0)
    
    std[std == 0.0] = 1.0

    X_scaled = (X_raw - mean) / std

    scaler_params = {
        "mean": mean,
        "std": std,
        "feature_names": feature_cols
    }

    np.random.seed(seed)
    n_samples = len(X_scaled)
    indices = np.random.permutation(n_samples)
    split_idx = int(n_samples * train_val_split)

    train_idx, val_idx = indices[:split_idx], indices[split_idx:]

    metadata_df["split"] = "train"
    metadata_df.iloc[val_idx, metadata_df.columns.get_loc("split")] = "val"

    device_obj = torch.device(device if torch.cuda.is_available() else "cpu")
    
    X_train_tensor = torch.tensor(X_scaled[train_idx], dtype=torch.float32, device=device_obj)
    X_val_tensor = torch.tensor(X_scaled[val_idx], dtype=torch.float32, device=device_obj)
    X_full_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=device_obj)

    print(X_train_tensor.shape)
    print(X_val_tensor.shape)
    print(X_full_tensor.shape)

    train_dataset = TensorDataset(X_train_tensor)
    val_dataset = TensorDataset(X_val_tensor)
    full_dataset = TensorDataset(X_full_tensor)

    train_loader = DataLoader(
        train_dataset, 
        batch_size=len(train_dataset), 
        shuffle=False
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=len(val_dataset), 
        shuffle=False
    )
    full_loader = DataLoader(
        full_dataset, 
        batch_size=len(full_dataset), 
        shuffle=False
    )

    return train_loader, val_loader, full_loader, metadata_df, scaler_params

In [ ]:
def train_autoencoder(
    model, 
    train_loader, 
    val_loader, 
    max_epochs=2000, 
    patience=40
):
    early_stop_callback = EarlyStopping(
        monitor="val_loss",
        min_delta=1e-5,
        patience=patience,
        mode="min",
        verbose=True
    )

    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        filename="best_vehicle_autoencoder"
    )

    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        callbacks=[early_stop_callback, checkpoint_callback],
        enable_progress_bar=True
    )

    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    
    return trainer, checkpoint_callback.best_model_path

In [ ]:
torch.set_float32_matmul_precision("high")

model = VehicleAutoencoder()
train, val, full, metadata, scaler = prepare_vehicle_data(df)
train_autoencoder(model, train, val)